# Grupo B · Áreas verdes e densidade populacional
**Oficina Python para Mapeamento Urbano** · Felipe Almeida ([felipe-almeida.com](https://felipe-almeida.com/))

Vocês vão cruzar a **população do Censo 2022 por setor censitário** com as **áreas verdes do OpenStreetMap** para discutir quem tem acesso a verde em BH.

### O que o grupo entrega (60 min)

| Tempo | Etapa | Produto |
|---|---|---|
| 10 min | Pergunta e hipótese | H1 e H0 escritas na seção 1 |
| 15 min | Conhecer e limpar o dado | tipos corrigidos, vazios tratados, recorte definido |
| 20 min | Um método **espacial** + um método **quantitativo** | tabela de resultados |
| 15 min | Mapa + gráfico com créditos | `outputs/grupo_X_mapa.png` e `outputs/grupo_X_grafico.png` |

No fim, cada grupo apresenta em 6 minutos: pergunta, método, mapa, gráfico e **um limite** da análise.

## 0. Ambiente

In [ ]:
!pip install -q geobr osmnx mapclassify folium
import os, sys
REPO = "https://github.com/brfelipealmeida/oficina-python-mapeamento"
if not os.path.exists("oficina-python-mapeamento"):
    !git clone -q {REPO}
sys.path.append("oficina-python-mapeamento/src")

import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from estilo import *
from dados import *
configurar_estilo()

## 1. Pergunta e hipótese

Escolham **uma** e reescrevam (ou criem outra):

- Setores mais densos estão mais longe de parques?
- A área verde a até 500 m por habitante é menor nas regionais periféricas?
- Setores em favelas e comunidades urbanas têm menos verde no entorno?

**H1:** _escreva aqui_
**H0:** _escreva aqui_
**Variável dependente:** _..._  **Variável independente:** _..._

## 2. Carregar os dados (pronto)

In [ ]:
setores = setores_bh()      # BH, Censo 2022
verde = osm_feicoes({"leisure": ["park", "garden", "nature_reserve"],
                     "landuse": ["forest", "grass", "recreation_ground"],
                     "natural": ["wood", "scrub", "grassland"]},
                    lugar="Belo Horizonte, Minas Gerais, Brazil")
verde = verde[verde.geom_type.isin(["Polygon", "MultiPolygon"])]
print(len(setores), "setores |", len(verde), "polígonos verdes")

## 3. Conhecer e limpar

1. Confiram `setores.dtypes`. Quais colunas são código e quais são número?
2. Quantos setores têm `pessoas` igual a 0 ou vazio? O que são? (parques, áreas industriais, aeroporto...)
3. O que é a coluna `NM_SUBDIST` em BH? (dica: compare com o nome das regionais)
4. Polígonos verdes muito pequenos (canteiros) entram ou não? Decidam um corte de área e justifiquem.

In [ ]:
setores[["CD_SETOR", "NM_SUBDIST", "NM_BAIRRO", "pessoas", "AREA_KM2", "em_favela"]].head()

In [ ]:
# TODO: tratar vazios e criar a densidade
setores["area_ha"] = setores.area / 10_000
setores["dens_ha"] = setores["pessoas"] / setores["area_ha"]
validos = setores[setores["pessoas"] > 0].copy()

# TODO: decidir o corte mínimo das áreas verdes (ex.: 0,5 ha = 5.000 m²)
verde["area_m2"] = verde.area
verde_f = verde[verde["area_m2"] >= 5_000].copy()
print(len(validos), "setores com população |", len(verde_f), "áreas verdes após o corte")

## 4. Método espacial (escolha um e complete)

**a) Distância ao verde mais próximo** (`sjoin_nearest`), do centróide do setor.
**b) Área verde no entorno** (`buffer` de 500 m do centróide + `overlay`), dividida pelos moradores.

In [ ]:
# a) pronto
cent = validos[["CD_SETOR", "geometry"]].copy()
cent["geometry"] = validos.centroid
prox = gpd.sjoin_nearest(cent, verde_f[["geometry"]], distance_col="dist_verde_m")
validos["dist_verde_m"] = validos["CD_SETOR"].map(prox.groupby("CD_SETOR")["dist_verde_m"].min())
validos["dist_verde_m"].describe().round(0)

In [ ]:
# b) TODO: área verde a até 500 m, por habitante
# entorno = cent.copy(); entorno["geometry"] = cent.buffer(500)
# inter = gpd.overlay(entorno, verde_f[["geometry"]].dissolve(), how="intersection")
# ...

## 5. Método quantitativo (escolha um)

| Opção | Quando usar | Código |
|---|---|---|
| Correlação | duas variáveis numéricas | `df["a"].corr(df["b"], method="spearman")` |
| Regressão | quanto Y muda quando X muda, com controles | `smf.ols("y ~ x + controle", data=df).fit()` |
| Comparação de grupos | uma variável numérica entre categorias | `stats.mannwhitneyu(g1, g2)` |

Lembre: p-valor pequeno indica que a diferença dificilmente é acaso; **não** indica que ela é grande ou importante. Olhe também o tamanho do efeito (diferença de medianas, coeficiente).

In [ ]:
print("Spearman densidade × distância ao verde:",
      round(validos["dens_ha"].corr(validos["dist_verde_m"], method="spearman"), 3))
modelo = smf.ols("np.log1p(dist_verde_m) ~ np.log1p(dens_ha) + em_favela", data=validos).fit()
print(modelo.summary().tables[1])

## 6. Mapa

In [ ]:
fig, ax = mapa_base((9, 10))
validos.plot(ax=ax, column="dist_verde_m", scheme="quantiles", k=5, cmap=CMAP, linewidth=0,
             legend=True, legend_kwds={"title": "metros até o verde", "fmt": "{:.0f}", "loc": "lower right"})
verde_f.plot(ax=ax, color=CORES["verde"], linewidth=0)
barra_escala(ax, 2000); seta_norte(ax)
titulo(ax, "TODO: título que responde à pergunta", "Distância do centro do setor à área verde mais próxima")
creditos(fig, fonte="IBGE, Censo 2022; OpenStreetMap", nota="Grupo B")
salvar(fig, "grupo_B_mapa")

## 7. Gráfico

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(validos["dens_ha"], validos["dist_verde_m"], s=6, alpha=0.4, color=CORES["roxo"])
ax.set_xscale("log")
ax.set_xlabel("Densidade (hab/ha, escala log)"); ax.set_ylabel("Distância ao verde (m)")
titulo(ax, "TODO: título", "Cada ponto é um setor censitário")
creditos(fig, fonte="IBGE, Censo 2022; OpenStreetMap", nota="Grupo B")
salvar(fig, "grupo_B_grafico")

## 8. Conclusão (escreva aqui)

1. **Resultado:** a hipótese foi sustentada, refutada ou ficou inconclusiva? Com que número?
2. **Leitura urbana:** o que isso significa para quem planeja a cidade?
3. **Limite:** o que o dado não mostra? (cobertura do OSM, escala do setor, distância em linha reta vs. caminho real, relevo...)